In [28]:
!pip install -r requirements.txt

  Using cached opentelemetry_semantic_conventions-0.60b1-py3-none-any.whl.metadata (2.4 kB)
  Using cached opentelemetry_api-1.39.1-py3-none-any.whl.metadata (1.5 kB)
Using cached opentelemetry_semantic_conventions-0.60b1-py3-none-any.whl (219 kB)
Using cached opentelemetry_api-1.39.1-py3-none-any.whl (66 kB)

  Attempting uninstall: opentelemetry-api

    Found existing installation: opentelemetry-api 1.42.1

    Uninstalling opentelemetry-api-1.42.1:

   ---------------------------------------- 0/2 [opentelemetry-api]
      Successfully uninstalled opentelemetry-api-1.42.1
   ---------------------------------------- 0/2 [opentelemetry-api]
   ---------------------------------------- 0/2 [opentelemetry-api]
  Attempting uninstall: opentelemetry-semantic-conventions
   ---------------------------------------- 0/2 [opentelemetry-api]
   ----------------- ----------------- 1/2 [opentelemetry-semantic-conventions]
    Found existing installation: opentelemetry-semantic-conventions 0.63b1


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-sdk 1.42.1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.39.1 which is incompatible.
opentelemetry-sdk 1.42.1 requires opentelemetry-semantic-conventions==0.63b1, but you have opentelemetry-semantic-conventions 0.60b1 which is incompatible.


# InvoFilter
## An invoice extractor automates the capture of billing data (like invoice numbers, totals, and line items) from documents. 
## - It eliminates need of manual data entry
## - It prevent human errors
## - It accelerate financial workflows
## ultimately saving hours of processing time and ensuring faster, more accurate vendor payments

### Import the library

In [49]:
import os
import pdfplumber
import json

### Function is created to extract data from pdf using pdfplumber

In [50]:
def extract_textfrompdf(path):
    rawtext = ""
    try:
        with pdfplumber.open(path) as pdf:
            for i, page in enumerate(pdf.pages):
                if page.extract_text():
                    rawtext += page.extract_text() + "\n"
            return rawtext
    except FileNotFoundError:
        print("error")

### Output of extract_textfrompdf function

In [52]:
ans=extract_textfrompdf("./invoices/invoice_1.pdf")
print(f"Extracted data from invoice_1- {ans}")

Extracted data from invoice_1- INVOICE
INVOICE #
INV-10012
FROM BILL TO
Your Plumbing Co. Your Client
1234 Your Street 1234 Clients Street
City, California City, California
90210 90210
United States United States
1-888-123-4567 1-888-123-8910
JOB DETAILS
JOB NUMBER SERVICE DATE DUE DATE SERVICE ADDRESS
JOB-2021-0847 26/3/2021 25/4/2021 Same as Bill To
DESCRIPTION QTY RATE AMOUNT
Service Callout Fee 1 $85.00 $85.00
Labor 3 $95.00 $285.00
Parts & Materials 1 $145.00 $145.00
Disposal Fee 1 $35.00 $35.00
Subtotal $550.00
Discount -$0.00
Tax +$44.00
TOTAL DUE $594.00
WORK PERFORMED
Diagnosed and repaired leaking kitchen sink. Replaced shutoff valve.
PAYMENT TERMS
Please pay within 30 days using the link in your invoice email.
LICENSE # PLUMB-000000
Invoicer.ai



In [53]:
lines=ans.split('\n')
print(f" lines - {lines}")
print("\n")
cleaned_lines = [line.strip() for line in lines if line.strip()]
print(f" cleaned_lines - {cleaned_lines}")
print("\n")
cleaned = "\n".join(cleaned_lines)
print(f" cleaned - {cleaned}")

 lines - ['INVOICE', 'INVOICE #', 'INV-10012', 'FROM BILL TO', 'Your Plumbing Co. Your Client', '1234 Your Street 1234 Clients Street', 'City, California City, California', '90210 90210', 'United States United States', '1-888-123-4567 1-888-123-8910', 'JOB DETAILS', 'JOB NUMBER SERVICE DATE DUE DATE SERVICE ADDRESS', 'JOB-2021-0847 26/3/2021 25/4/2021 Same as Bill To', 'DESCRIPTION QTY RATE AMOUNT', 'Service Callout Fee 1 $85.00 $85.00', 'Labor 3 $95.00 $285.00', 'Parts & Materials 1 $145.00 $145.00', 'Disposal Fee 1 $35.00 $35.00', 'Subtotal $550.00', 'Discount -$0.00', 'Tax +$44.00', 'TOTAL DUE $594.00', 'WORK PERFORMED', 'Diagnosed and repaired leaking kitchen sink. Replaced shutoff valve.', 'PAYMENT TERMS', 'Please pay within 30 days using the link in your invoice email.', 'LICENSE # PLUMB-000000', 'Invoicer.ai', '']


 cleaned_lines - ['INVOICE', 'INVOICE #', 'INV-10012', 'FROM BILL TO', 'Your Plumbing Co. Your Client', '1234 Your Street 1234 Clients Street', 'City, California Cit

### Function to clean raw data extracted from pdf

In [54]:
def cleantext(text):
    # Step 1 — Split the text into individual lines
    lines=text.split('\n')
    # Step 2 — Strip whitespace from each line AND remove empty lines
    cleaned_lines = [line.strip() for line in lines if line.strip()]
    # Step 3 — Join the cleaned lines back into one single string
    cleaned = "\n".join(cleaned_lines)
    return cleaned

### Class to generate required fields from data using Mistral AI 

In [55]:
import os
from mistralai.client import Mistral
class DataExtraction:
    def __init__(self):
        # 1. Fetch the API key when the class is initialized
        self.api_key = os.getenv("MISTRAL_API_KEY")
        self.client = Mistral(api_key=self.api_key)
        print("Key found:", api_key is not None)
    def generateresponse(self,system_prompt,cleaned_invoice,temp=0):
        response = self.client.chat.complete(
        model="mistral-large-latest",
        response_format={"type": "json_object"}, 
        messages=[
        {
            "role": "system",
            "content": system_prompt          # our extraction instructions
        },
        {
            "role": "user",
            "content": f"Here is the document text to extract from:\n\n{cleaned_invoice}"                              
        }
        ],
        temperature=temp                             
        )
        # Extract the text content from the response
        raw_response = response.choices[0].message.content
        print("✅ Response received")
        return raw_response

### System prompt

In [56]:
system_prompt = """You are a precise document data extractor.
Your task is to extract specific information from the document text provided by the user.
Return your response as a valid JSON object with EXACTLY these fields:
{
"invoice_number":"The unique invoice ID or reference number",
"invoice_date":"Date the invoice was issued — format DD-MM-YYYY",
"due_date":"Payment deadline — null if not mentioned",
"billed_by":"Name of the company or person issuing the invoice",
"billed_to":"Name of the client or recipient being billed",
"line_items":"line_items	List of services or products with amounts	[{item, amount}]",
"subtotal":"Total before any discounts or taxes",
"discount":"Any discount applied — null if none",
"tax_or_gst":"Tax amount if mentioned — null if not applicable",
"total_amount":"The final payable amount",
"currency:"Currency used in the invoice",
"payment_method":"How payment should be made — null if not mentioned	NEFT / UPI",
"notes":"Any additional remarks or instructions on the invoice"
}
Rules:
- Return ONLY the JSON object. No explanation. No markdown. No code blocks.
- If a field cannot be found in the document, set its value to null.
- Do not guess or make up values. Only extract what is clearly present.
- Convert the due_date and invoice_date to the format DD-MM-YYYY.
"""

print("✅ System prompt ready.")


✅ System prompt ready.


### Code to extract required fields

In [57]:
import json
from dotenv import load_dotenv
load_dotenv(dotenv_path=".env")
ans=extract_textfrompdf("./invoices/invoice_1.pdf")
cleaned_invoice=cleantext(ans)
deobj=DataExtraction()
deresult=deobj.generateresponse(system_prompt,cleaned_invoice)
jd=json.loads(deresult)
print(f"JSON data = {jd}")

Key found: True
✅ Response received
JSON data = {'invoice_number': 'INV-10012', 'invoice_date': '26-03-2021', 'due_date': '25-04-2021', 'billed_by': 'Your Plumbing Co.', 'billed_to': 'Your Client', 'line_items': [{'item': 'Service Callout Fee', 'amount': 85.0}, {'item': 'Labor', 'amount': 285.0}, {'item': 'Parts & Materials', 'amount': 145.0}, {'item': 'Disposal Fee', 'amount': 35.0}], 'subtotal': 550.0, 'discount': 0.0, 'tax_or_gst': 44.0, 'total_amount': 594.0, 'currency': 'USD', 'payment_method': None, 'notes': 'Diagnosed and repaired leaking kitchen sink. Replaced shutoff valve. Please pay within 30 days using the link in your invoice email.'}


## Now we will extract required fields from 3 invoice pdfs and store it in output files

In [59]:
import json
import time  # 1. Required for the delay
from dotenv import load_dotenv
load_dotenv()
deobj=DataExtraction()
for i in range(1,4):
    
    ans=extract_textfrompdf(f"./invoices/invoice_{i}.pdf")
    
    cleaned_invoice=cleantext(ans)
    
    deresult=deobj.generateresponse(system_prompt,cleaned_invoice)
    
    print(f"Output of invoice_{i} - {json.loads(deresult)}")
    print("\n")
    time.sleep(5)
    
    with open(f"./outputs/output_invoice_{i}.json", "w") as f:
        jd=json.loads(deresult)
        json.dump(jd,f,indent=4) 
        print(f"Data is stored in file output_invoice_{i}.json")

Key found: True
✅ Response received
Output of invoice_1 - {'invoice_number': 'INV-10012', 'invoice_date': '26-03-2021', 'due_date': '25-04-2021', 'billed_by': 'Your Plumbing Co.', 'billed_to': 'Your Client', 'line_items': [{'item': 'Service Callout Fee', 'amount': 85.0}, {'item': 'Labor', 'amount': 285.0}, {'item': 'Parts & Materials', 'amount': 145.0}, {'item': 'Disposal Fee', 'amount': 35.0}], 'subtotal': 550.0, 'discount': 0.0, 'tax_or_gst': 44.0, 'total_amount': 594.0, 'currency': 'USD', 'payment_method': None, 'notes': 'Diagnosed and repaired leaking kitchen sink. Replaced shutoff valve. Please pay within 30 days using the link in your invoice email.'}


Data is stored in file output_invoice_1.json
✅ Response received
Output of invoice_2 - {'invoice_number': 'TEST01', 'invoice_date': '15-06-2020', 'due_date': None, 'billed_by': 'Amy Fare', 'billed_to': 'Jane Doe', 'line_items': [{'item': 'Tutoring (1 hr)', 'amount': 20.0}], 'subtotal': 20.0, 'discount': 5.0, 'tax_or_gst': None, 't